In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import HistGradientBoostingRegressor, StackingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import BaggingRegressor
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import logging
from sklearn.linear_model import ElasticNet

In [3]:
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def bias(y_true, y_pred) -> float:
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))

In [4]:
%run 05_0.1_visualization_helpers.ipynb  


c:\Users\Rosa Melo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# lists previously defined in 05_0_preproc_helpers.ipynb
numeric_features = num_feat                      # ['year', 'mileage', ...]
categorical_features = cat_feat                  # ['Brand', 'model', 'transmission', 'fuelType']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

N_SPLITS = 8 # the number of folds for K-Fold cross-validation
RANDOM_STATE = 42 # seed to control randomness

X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


modelo normal, features todas, sem fe/fs price normal

In [ ]:
# K-Fold cross-validation setup.
# Shuffling ensures each fold is representative of the full dataset,
# and the fixed random_state guarantees reproducibility.

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Fixed hyperparameters for all base models.
# These were previously selected and are kept constant across folds
# to ensure a fair and stable evaluation.

params = {
    "hgb": {
        "max_iter": 600,
        "learning_rate": 0.1,
        "max_depth": 20,
        "max_leaf_nodes": 127,
        "min_samples_leaf": 20,
        "l2_regularization": 0,
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 1000,
        "min_samples_split": 6,
        "min_samples_leaf": 1,
        "max_features": None,
        "max_depth": 15,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 1200,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 0.7,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

# Logging setup to persist fold-by-fold and final CV results.
# This allows offline inspection and comparison between runs.
log_path = "stacking_complete_results.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =========================================")
    log("# SINGLE CONFIGURATION CROSS-VALIDATION")
    log("# ENSEMBLE: HGB + RF + ET (STACKING)")
    log("# =========================================")
    log(f"Parameters: {params}")
    log("")

    # Containers for tracking metrics across folds
    fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
    fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
    fold_med_ae = []

    # Main K-Fold loop
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

        log(f"\n===== FOLD {fold}/{N_SPLITS} =====")

        # Split data into train and validation folds
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

        
        # Feature preprocessing
        
        # All preprocessing steps are fit only on the training fold
        # and then applied to validation, preventing data leakage.

        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, year_state)
        X_val   = transform_year_with_model_median(X_val, year_state)

        mileage_state = fit_mileage_imputer(X_train, "mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, mileage_state)
        X_val   = transform_mileage_imputer(X_val, mileage_state)

        engine_state = fit_engine_size_imputer(X_train, "engineSize")
        X_train = transform_engine_size_imputer(X_train, engine_state)
        X_val   = transform_engine_size_imputer(X_val, engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, "mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, mpg_state)
        X_val   = transform_mpg_imputer(X_val, mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, owners_state)
        X_val   = transform_previous_owners_imputer(X_val, owners_state)

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val, _, _   = transform_invalid_models(X_val, model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val, _, _   = transform_transmission_resolver(X_val, transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

        
        # Encoding
        
        # High-cardinality features are target-encoded using only
        # the training fold targets to avoid leakage.
        # Low-cardinality features use one-hot encoding.
        high_card_features = ["Brand", "model"]
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])
        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        X_train_final = pd.concat([X_train[numeric_features], X_train_high, X_train_low], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features],   X_val_high,   X_val_low], axis=1)

        
        # Scaling
        
        # Scaling is required for the linear meta-model (Ridge).
        # The scaler is fit on training data only.
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_final)
        X_val_scaled   = scaler.transform(X_val_final)

       
        # Stacking model
       
        # Tree-based models act as base learners.
        # Ridge regression is used as a regularized meta-model
        # to combine base predictions and reduce overfitting.

        estimators = [
            ("hgb", HistGradientBoostingRegressor(**params["hgb"])),
            ("rf",  RandomForestRegressor(**params["rf"])),
            ("et",  ExtraTreesRegressor(**params["et"]))
        ]

        meta = Ridge(alpha=4.0)

        stack = StackingRegressor(
            estimators=estimators,
            final_estimator=meta,
            cv=5,
            passthrough=False,
            n_jobs=1
        )

        stack.fit(X_train_scaled, y_train)

        
        # PREDICTIONS
       
        y_tr = stack.predict(X_train_scaled)
        y_val_pred = stack.predict(X_val_scaled)

      
        # Metrics
        
        # Metrics are computed on the original target scale
        # to preserve interpretability.
        mae_tr  = mean_absolute_error(y_train, y_tr)
        rmse_tr = np.sqrt(mean_squared_error(y_train, y_tr))
        r2_tr   = r2_score(y_train, y_tr)
        bias_tr = np.mean(y_train - y_tr)

        mae_val  = mean_absolute_error(y_val, y_val_pred)
        rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
        r2_val   = r2_score(y_val, y_val_pred)
        bias_val = np.mean(y_val - y_val_pred)
        med_ae   = median_absolute_error(y_val, y_val_pred)

        log(f"[TRAIN] R2={r2_tr:.4f} | RMSE={rmse_tr:.0f} | MAE={mae_tr:.0f} | Bias={bias_tr:.1f}")
        log(f"[VAL]   R2={r2_val:.4f} | RMSE={rmse_val:.0f} | MAE={mae_val:.0f} | Bias={bias_val:.1f}")

        fold_maes_tr.append(mae_tr)
        fold_rmses_tr.append(rmse_tr)
        fold_r2s_tr.append(r2_tr)
        fold_bias_tr.append(bias_tr)

        fold_maes_val.append(mae_val)
        fold_rmses_val.append(rmse_val)
        fold_r2s_val.append(r2_val)
        fold_bias_val.append(bias_val)
        fold_med_ae.append(med_ae)

     # Final aggregated CV results
    log("\n===== FINAL CROSS-VALIDATION RESULTS =====")
    log(f"[TRAIN AVG] MAE={np.mean(fold_maes_tr):.1f} | R2={np.mean(fold_r2s_tr):.4f} | Bias={np.mean(fold_bias_tr):.1f}")
    log(f"[VAL AVG]   MAE={np.mean(fold_maes_val):.1f} | "
        f"R2={np.mean(fold_r2s_val):.4f} | "
        f"Bias={np.mean(fold_bias_val):.1f} | "
        f"RMSE={np.mean(fold_rmses_val):.1f}")


# =========================================
# SINGLE CONFIGURATION CROSS-VALIDATION
# ENSEMBLE: HGB + RF + ET (STACKING)
# =========================================
Parameters: {'hgb': {'max_iter': 600, 'learning_rate': 0.1, 'max_depth': 20, 'max_leaf_nodes': 127, 'min_samples_leaf': 20, 'l2_regularization': 0, 'random_state': 42}, 'rf': {'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True, 'random_state': 42, 'n_jobs': -1}, 'et': {'n_estimators': 1200, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.7, 'bootstrap': False, 'random_state': 42, 'n_jobs': -1}}


===== FOLD 1/8 =====
[TRAIN] R2=0.9798 | RMSE=1387 | MAE=890 | Bias=-0.5
[VAL]   R2=0.9583 | RMSE=1942 | MAE=1252 | Bias=-45.5

===== FOLD 2/8 =====
[TRAIN] R2=0.9804 | RMSE=1365 | MAE=901 | Bias=-2.0
[VAL]   R2=0.9541 | RMSE=2074 | MAE=1269 | Bias=-16.7

===== FOLD 3/8 =====
[TRAIN] R2=0.9803 | RMSE=1367 | MAE=888 | Bias=1.

o modelo base mas sem previous owners, log do price

In [ ]:
# K-Fold cross-validation setup.
# Shuffling improves fold representativeness and the fixed random_state
# guarantees reproducibility across runs.
kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Feature explicitly removed from the model.
# This allows measuring the impact of excluding potentially noisy or biased variables.
DROP_FROM_MODEL = ["previousOwners"]

# Numerical features used by the model.
# Includes 'year', which captures vehicle age and is a key price determinant.
numeric_features = ["mileage", "engineSize", "tax", "mpg", "year"]

# Fixed hyperparameters for all base learners.
# These were selected beforehand and kept constant across folds
# to ensure a fair and controlled comparison.
params = {
    "hgb": {
        "max_iter": 1000,
        "learning_rate": 0.1,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 20,
        "l2_regularization": 3.0,
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 1000,
        "min_samples_split":2,
        "min_samples_leaf": 2,
        "max_features": 0.33,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 800,
        "max_depth": 20,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.7,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

# Logging configuration to store fold-level and final CV results.
# This enables post-analysis and comparison between experiments.
log_path = "stacking_no_previous_cv_results.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# ===============================================")
    log("# SINGLE CONFIG CV — WITHOUT previousOwners")
    log("# ENSEMBLE: HGB + RF + ET (STACKING)")
    log("# ===============================================")
    log(f"Parameters: {params}")
    log("")

    # Containers to aggregate metrics across folds
    fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
    fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
    fold_med_ae = []

    
    # Main K-Fold loop
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

        log(f"\n===== FOLD {fold}/{N_SPLITS} =====")

        # Split data into training and validation folds
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

        # Log-transform the target to stabilize variance
        # and reduce the influence of extreme prices.
        y_train_log = np.log1p(y_train)

        
        # Preprocessing
       
        # All preprocessing steps are fit exclusively on the training fold
        # and then applied to validation, preventing information leakage.
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, year_state)
        X_val   = transform_year_with_model_median(X_val, year_state)

        mileage_state = fit_mileage_imputer(X_train, "mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, mileage_state)
        X_val   = transform_mileage_imputer(X_val, mileage_state)

        engine_state = fit_engine_size_imputer(X_train, "engineSize")
        X_train = transform_engine_size_imputer(X_train, engine_state)
        X_val   = transform_engine_size_imputer(X_val, engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, "mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, mpg_state)
        X_val   = transform_mpg_imputer(X_val, mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, owners_state)
        X_val   = transform_previous_owners_imputer(X_val, owners_state)

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val, _, _   = transform_invalid_models(X_val, model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val, _, _   = transform_transmission_resolver(X_val, transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

        # Explicitly remove previousOwners from the model input
        # to assess performance without this feature.
        X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
        X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

        # Encoding
        
        # High-cardinality categorical features are target-encoded
        # using only training-fold targets to avoid leakage.
        # Low-cardinality features are one-hot encoded.
        high_card_features = ["Brand", "model"]
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])
        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        X_train_final = pd.concat(
            [X_train[numeric_features], X_train_high, X_train_low], axis=1
        )
        X_val_final = pd.concat(
            [X_val[numeric_features], X_val_high, X_val_low], axis=1
        )

       
        # Scaling
    
        # Feature scaling is required for the linear meta-model.
        # The scaler is fit only on training data.
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_final)
        X_val_scaled   = scaler.transform(X_val_final)

        if fold == 1:
            log(f"  > Features Used ({X_train_final.shape[1]})")

       
        # Stacking model
       
        # Tree-based models act as base learners.
        # Ridge regression serves as a regularized meta-model,
        # combining base predictions while controlling overfitting.
        estimators = [
            ("hgb", HistGradientBoostingRegressor(**params["hgb"])),
            ("rf",  RandomForestRegressor(**params["rf"])),
            ("et",  ExtraTreesRegressor(**params["et"]))
        ]

        meta_model = Ridge(alpha=3.0)

        stack = StackingRegressor(
            estimators=estimators,
            final_estimator=meta_model,
            cv=5,
            n_jobs=1
        )


        # Model is trained on log-transformed targets
        
        stack.fit(X_train_scaled, y_train_log)

        # Predictions are transformed back to the original price scale

        pred_tr  = np.expm1(stack.predict(X_train_scaled))
        pred_val = np.expm1(stack.predict(X_val_scaled))
        
        
        # Metrics
        
        # All evaluation metrics are computed on the original price scale
        # for interpretability.

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        rmse_tr = np.sqrt(mean_squared_error(y_train, pred_tr))
        r2_tr   = r2_score(y_train, pred_tr)
        bias_tr = np.mean(y_train - pred_tr)

        mae_val  = mean_absolute_error(y_val, pred_val)
        rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))
        r2_val   = r2_score(y_val, pred_val)
        bias_val = np.mean(y_val - pred_val)
        med_ae   = median_absolute_error(y_val, pred_val)

        log(f"[TRAIN] R2={r2_tr:.4f} | RMSE={rmse_tr:.0f} | MAE={mae_tr:.0f} | Bias={bias_tr:.1f}")
        log(f"[VAL]   R2={r2_val:.4f} | RMSE={rmse_val:.0f} | MAE={mae_val:.0f} | Bias={bias_val:.1f}")

        fold_maes_tr.append(mae_tr)
        fold_rmses_tr.append(rmse_tr)
        fold_r2s_tr.append(r2_tr)
        fold_bias_tr.append(bias_tr)

        fold_maes_val.append(mae_val)
        fold_rmses_val.append(rmse_val)
        fold_r2s_val.append(r2_val)
        fold_bias_val.append(bias_val)
        fold_med_ae.append(med_ae)

   # Final aggregated cross-validation results

    log("\n===== FINAL CROSS-VALIDATION RESULTS =====")
    log(f"[TRAIN AVG] MAE={np.mean(fold_maes_tr):.1f} | "
        f"R2={np.mean(fold_r2s_tr):.4f} | Bias={np.mean(fold_bias_tr):.1f}")
    log(f"[VAL AVG]   MAE={np.mean(fold_maes_val):.1f} | "
        f"R2={np.mean(fold_r2s_val):.4f} | "
        f"Bias={np.mean(fold_bias_val):.1f} | "
        f"RMSE={np.mean(fold_rmses_val):.1f}")


# ===============================================
# SINGLE CONFIG CV — WITHOUT previousOwners
# ENSEMBLE: HGB + RF + ET (STACKING)
# ===============================================
Parameters: {'hgb': {'max_iter': 1000, 'learning_rate': 0.1, 'max_depth': 20, 'max_leaf_nodes': 191, 'min_samples_leaf': 20, 'l2_regularization': 3.0, 'random_state': 42}, 'rf': {'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.33, 'max_depth': 20, 'bootstrap': True, 'random_state': 42, 'n_jobs': -1}, 'et': {'n_estimators': 800, 'max_depth': 20, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.7, 'bootstrap': False, 'random_state': 42, 'n_jobs': -1}}


===== FOLD 1/8 =====
  > Features Used (15)
[TRAIN] R2=0.9751 | RMSE=1540 | MAE=887 | Bias=54.1
[VAL]   R2=0.9604 | RMSE=1894 | MAE=1208 | Bias=39.7

===== FOLD 2/8 =====
[TRAIN] R2=0.9768 | RMSE=1483 | MAE=908 | Bias=55.0
[VAL]   R2=0.9538 | RMSE=2081 | MAE=1242 | Bias=76.6

===== FOLD 3/8 =====
[TRAIN] R2

feature engineering com tudo, sem selection 

In [ ]:

#LOGGING

# Logging configuration to track fold-level execution and results.
# This allows full reproducibility and post analysis of model behavior.

LOG_FILE = "stacking_fe_no_fs_.log"

# Initial numeric feature set before feature engineering.
# Some of these variables will later be transformed or replaced.
numeric_features = ["mileage", "engineSize", "tax", "mpg", "year", "previousOwners"]

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)

logging.info(f"SINGLE CONFIG | {N_SPLITS}-fold | HGB + RF + ET | FE | NO FS")


# SINGLE CONFIGURATION

# Identifier for this specific experimental setup.
CONFIG_NAME = "HGB_RF_ET_FE_NO_FS"

# Fixed hyperparameters for each base learner.
# These parameters remain constant across folds to ensure
# a controlled and unbiased cross-validation evaluation.
PARAMS = {
    "hgb": {
        "max_iter": 800,
        "learning_rate": np.float64(0.08736842105263158),
        "max_depth": 15,
        "max_leaf_nodes": 127,
        "min_samples_leaf": 20,
        "l2_regularization": np.float64(1.236842105263158),
        "loss": "squared_error",
        "max_bins": 255,
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 200,
        "min_samples_split":10,
        "min_samples_leaf": 2,
        "max_features": None,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 800,
        "max_depth": 20,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.7,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}


# K-FOLD CONFIGURATION

# Standard K-Fold cross-validation with shuffling to ensure
# each fold is representative of the full dataset distribution.
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Categorical columns that require string normalization
# to reduce artificial category fragmentation.
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

# High-cardinality categorical features (target encoded).
high_card_features = ["Brand", "model"]
# Low-cardinality categorical features (one-hot encoded).
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalizes categorical string columns by:
    - Filling missing values with a consistent placeholder
    - Standardizing casing and spacing
    This reduces noise caused by inconsistent raw text entries.
    """
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df,
                column=col,
                remove_middle_spaces=True,
                allow_extra_chars=""
            )
    return df


# SINGLE CONFIGURATION EVALUATION

def eval_single_config(name: str, params: dict) -> dict:
    """
    Evaluates a single model configuration using K-Fold CV.
    All preprocessing, feature engineering, encoding, and scaling
    are fit strictly on the training fold to avoid leakage.
    """
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"[{name}] Fold {fold}/{N_SPLITS}")
        logging.info(f"[{name}] Fold {fold}/{N_SPLITS}")

        # Split data into training and validation folds
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # Log-transform the target to stabilize variance
        # and reduce the impact of extreme price values.
        y_train_log = np.log1p(y_train)

        # Normalize categorical string columns
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restrict dataset to expected base columns
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # Explicit validation to ensure required columns
        # are present before continuing the pipeline.
        required = ["year", "mileage", "engineSize", "tax", "mpg", "previousOwners", "Brand", "model"]
        missing = [c for c in required if c not in X_train.columns]
        if missing:
            raise KeyError(f"[Fold {fold}] faltam colunas necessárias no pipeline: {missing}")

        
        # DATA CLEANING
       
        # All imputers are fit on the training fold only.

        year_state = fit_year_median(X_train, "year", "model")
        X_train = transform_year_with_model_median(X_train, year_state)
        X_val   = transform_year_with_model_median(X_val,   year_state)

        mileage_state = fit_mileage_imputer(X_train, "mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, mileage_state)
        X_val   = transform_mileage_imputer(X_val,   mileage_state)

        engine_state = fit_engine_size_imputer(X_train, "engineSize")
        X_train = transform_engine_size_imputer(X_train, engine_state)
        X_val   = transform_engine_size_imputer(X_val,   engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, "mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, mpg_state)
        X_val   = transform_mpg_imputer(X_val,   mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   owners_state)

       
        # CATEGORY RESOLUTION
        
        # Domain-driven rules to correct invalid or ambiguous labels.

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        
        # FEATURE ENGINEERING
        
        # Explicit feature engineering to encode domain knowledge.

        # Converts number of owners into a binary signal and removes the raw count.
        X_train = add_owners_flagged(X_train, "previousOwners", "owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   "previousOwners", "owners_flagged", drop_original=True)

        # Replaces 'year' with vehicle age, a more meaningful representation.
        X_train = create_age_and_drop_year(X_train, "year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   "year", base_year=2020)

        # Normalizes mileage relative to vehicle age.
        X_train = add_mileage_features(X_train, "mileage", "age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   "mileage", "age", drop_original=True, drop_ratio=True)

        # Discretizes engine size into bins to capture non-linear effects.
        X_train = add_engine_bins(X_train, "engineSize", "engine_bin")
        X_val   = add_engine_bins(X_val,   "engineSize", "engine_bin")

        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        
        # D) ENCODING
        
        # Target encoding for high-cardinality features using
        # training-fold targets only (log scale).
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        # One-hot encoding for low-cardinality categorical features.
        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low], axis=1)

        # Numeric columns are those not involved in categorical encoding.
        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_curr],   X_val_cat], axis=1)

        # Scale numeric features only.
        scaler = StandardScaler()
        X_train_final[numeric_curr] = scaler.fit_transform(X_train_final[numeric_curr])
        X_val_final[numeric_curr]   = scaler.transform(X_val_final[numeric_curr])

        # Align validation columns to training columns.
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

       
        # MODELS + ENSEMBLE
       
        estimators = [
            ("hgb", HistGradientBoostingRegressor(**params["hgb"])),
            ("rf",  RandomForestRegressor(**params["rf"])),
            ("et",  ExtraTreesRegressor(**params["et"]))
        ]

        # Ridge regression is used as a regularized meta-learner.
        meta_model = Ridge(alpha=3.0)

        stack = StackingRegressor(
            estimators=estimators,
            final_estimator=meta_model,
            cv=5,
            n_jobs=1
        )

        # Train on log-transformed targets.
        stack.fit(X_train_final, y_train_log)

        # Predictions are transformed back to original price scale.
        pred_tr  = np.expm1(stack.predict(X_train_final))
        pred_val = np.expm1(stack.predict(X_val_final))

        
        # METRICS
        fold_rows.append({
            "fold": fold,
            "n_features": X_train_final.shape[1],
            "train_rmse": rmse(y_train, pred_tr),
            "val_rmse":   rmse(y_val,   pred_val),
            "train_mae":  mean_absolute_error(y_train, pred_tr),
            "val_mae":    mean_absolute_error(y_val,   pred_val),
            "train_r2":   r2_score(y_train, pred_tr),
            "val_r2":     r2_score(y_val,   pred_val),
            "train_bias": bias(y_train, pred_tr),
            "val_bias":   bias(y_val,   pred_val),
        })

    df = pd.DataFrame(fold_rows)

    # Aggregate cross-validation metrics
    return {
        "config": name,
        "val_rmse_mean": df["val_rmse"].mean(),
        "val_mae_mean":  df["val_mae"].mean(),
        "val_r2_mean":   df["val_r2"].mean(),
        "val_bias_mean": df["val_bias"].mean(),
        "train_rmse_mean": df["train_rmse"].mean(),
        "train_mae_mean":  df["train_mae"].mean(),
        "train_r2_mean":   df["train_r2"].mean(),
        "train_bias_mean": df["train_bias"].mean(),
        "avg_features": df["n_features"].mean(),
    }


# RUN

print("\n-- Running single configuration --")
result = eval_single_config(CONFIG_NAME, PARAMS)

results_df = pd.DataFrame([result])
display(results_df)

# Save aggregated results to disk
OUT_CSV = "mean_fe_no_fs.csv"
results_df.to_csv(OUT_CSV, index=False)

print("\nFINAL RESULTS")
print(f"VAL RMSE: {result['val_rmse_mean']:.2f}")
print(f"VAL R2:   {result['val_r2_mean']:.4f}")
print(f"VAL MAE:  {result['val_mae_mean']:.2f}")
print(f"VAL Bias: {result['val_bias_mean']:.2f}")
print(f"Avg Features: {result['avg_features']:.0f}")

print(f"\nResults saved to: {OUT_CSV}")
print(f"Log file: {LOG_FILE}")



-- Running single configuration --
[HGB_RF_ET_FE_NO_FS] Fold 1/8
[HGB_RF_ET_FE_NO_FS] Fold 2/8
[HGB_RF_ET_FE_NO_FS] Fold 3/8
[HGB_RF_ET_FE_NO_FS] Fold 4/8
[HGB_RF_ET_FE_NO_FS] Fold 5/8
[HGB_RF_ET_FE_NO_FS] Fold 6/8
[HGB_RF_ET_FE_NO_FS] Fold 7/8
[HGB_RF_ET_FE_NO_FS] Fold 8/8


,config,val_rmse_mean,val_mae_mean,val_r2_mean,val_bias_mean,train_rmse_mean,train_mae_mean,train_r2_mean,train_bias_mean,avg_features
0,HGB_RF_ET_FE_NO_FS,2141.518868,1248.727249,0.951417,93.512092,1512.507268,922.976454,0.975856,57.149877,25.0



FINAL RESULTS
VAL RMSE: 2141.52
VAL R2:   0.9514
VAL MAE:  1248.73
VAL Bias: 93.51
Avg Features: 25

Results saved to: mean_fe_no_fs.csv
Log file: stacking_fe_no_fs_.log


fe com fs, 65 percentagem

In [ ]:
# SINGLE CONFIG — FEATURE ENGINEERING + FEATURE SELECTION + STACKING

# This experiment evaluates a full pipeline combining:
# - domain-driven feature engineering
# - embedded feature selection
# - stacking ensemble learning

FS_KEEP_RATIO = 0.65
CONFIG_NAME = "FE_FS65_Stacking"

# Initial numeric features before feature engineering.
numeric_features = ["mileage", "engineSize", "tax", "mpg", "year", "previousOwners"]


# FEATURE SELECTION MODEL (RANDOM FOREST)

# A Random Forest is used as an embedded feature selector
# based on feature importance scores learned from the training fold only.
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}


# BASE MODEL PARAMETERS (FIXED)

# Hyperparameters are fixed to ensure a fair and unbiased
# cross-validation evaluation.
PARAMS = {
    "hgb": {
        "max_iter": 1000,
        "learning_rate": np.float64(0.12052631578947368),
        "max_depth": 25,
        "max_leaf_nodes": 127,
        "min_samples_leaf": 20,
        "l2_regularization": np.float64(2.394736842105263),
        "max_bins": 255,
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 1000,
        "min_samples_split":2,
        "min_samples_leaf": 2,
        "max_features": 0.33,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 800,
        "max_depth": 20,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.7,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}


# LOGGING

# Logging fold-level execution for traceability and reproducibility.

LOG_FILE = "stacking_fe_fs_65.log"
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info(f"{CONFIG_NAME} | FS_KEEP_RATIO={FS_KEEP_RATIO} | {N_SPLITS}-fold")


# K-FOLD CROSS-VALIDATION

# Shuffled K-Fold ensures each fold is representative
# of the overall data distribution.
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Categorical columns requiring string normalization
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

# Categorical columns requiring string normalization
high_card_features = ["Brand", "model"]
# Low-cardinality features are one-hot encoded
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df):
    """
    Normalizes categorical string columns to reduce
    artificial category fragmentation due to formatting inconsistencies.
    """
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(df, col, True, "")
    return df


# SINGLE CONFIGURATION EVALUATION

def eval_single_config(name: str, params: dict) -> dict:
    rows = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"[{name}] Fold {fold}/{N_SPLITS}")

        # Split data into training and validation folds
        X_train = X.iloc[tr_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[tr_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # Log-transform the target to stabilize variance
        y_train_log = np.log1p(y_train)

        # Normalize categorical strings
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restrict to required base columns
        base_cols = [c for c in numeric_features + categorical_features if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # DATA CLEANING

        # All imputers are fit on the training fold only
        # to avoid information leakage.
        year_state = fit_year_median(X_train, "year", "model")
        X_train = transform_year_with_model_median(X_train, year_state)
        X_val   = transform_year_with_model_median(X_val,   year_state)

        mileage_state = fit_mileage_imputer(X_train, "mileage", True)
        X_train = transform_mileage_imputer(X_train, mileage_state)
        X_val   = transform_mileage_imputer(X_val,   mileage_state)

        engine_state = fit_engine_size_imputer(X_train, "engineSize")
        X_train = transform_engine_size_imputer(X_train, engine_state)
        X_val   = transform_engine_size_imputer(X_val,   engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, "mpg", True)
        X_train = transform_mpg_imputer(X_train, mpg_state)
        X_val   = transform_mpg_imputer(X_val,   mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   owners_state)

        # CATEGORY RESOLUTION
        # Domain-based rules to correct ambiguous or invalid labels.

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

       # FEATURE ENGINEERING 
        # Explicit encoding of domain knowledge.

        X_train = add_owners_flagged(X_train, "previousOwners", "owners_flagged", True)
        X_val   = add_owners_flagged(X_val,   "previousOwners", "owners_flagged", True)

        X_train = create_age_and_drop_year(X_train, "year", 2020)
        X_val   = create_age_and_drop_year(X_val,   "year", 2020)

        X_train = add_mileage_features(X_train, "mileage", "age", True, True)
        X_val   = add_mileage_features(X_val,   "mileage", "age", True, True)

        X_train = add_engine_bins(X_train, "engineSize", "engine_bin")
        X_val   = add_engine_bins(X_val,   "engineSize", "engine_bin")

        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # ENCODING
        # Target encoding is fit on training data only
        # using the log-transformed target.

        te = MyTargetEncoder(5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low], axis=1)

        drop_cols = set(high_card_features + low_card_curr)
        num_cols = [c for c in X_train.columns if c not in drop_cols]

        X_train_final = pd.concat([X_train[num_cols], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[num_cols],   X_val_cat], axis=1)

        # Standardize numeric features only
        scaler = StandardScaler()
        X_train_final[num_cols] = scaler.fit_transform(X_train_final[num_cols])
        X_val_final[num_cols]   = scaler.transform(X_val_final[num_cols])

        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # FEATURE SELECTION 
        # Embedded feature selection using Random Forest importance scores.
        # The selector is fit on the training fold only.

        n_feats = X_train_final.shape[1]
        k = max(1, int(np.ceil(FS_KEEP_RATIO * n_feats)))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            rf_fs,
            max_features=k,
            threshold=-np.inf,
            prefit=True
        )

        cols_sel = X_train_final.columns[selector.get_support()]
        X_train_sel = X_train_final[cols_sel]
        X_val_sel   = X_val_final[cols_sel]

        # STACKING ENSEMBLE 
        estimators = [
            ("hgb", HistGradientBoostingRegressor(**params["hgb"])),
            ("rf",  RandomForestRegressor(**params["rf"])),
            ("et",  ExtraTreesRegressor(**params["et"]))
        ]

        # Ridge regression acts as a regularized meta-learner.
        meta_model = Ridge(alpha=3.0)

        stack = StackingRegressor(
            estimators=estimators,
            final_estimator=meta_model,
            cv=5,
            n_jobs=1
        )

        stack.fit(X_train_sel, y_train_log)

        # Predictions are transformed back to the original price scale.
        pred_tr  = np.expm1(stack.predict(X_train_sel))
        pred_val = np.expm1(stack.predict(X_val_sel))

        rows.append({
            "fold": fold,
            "n_total": n_feats,
            "n_selected": len(cols_sel),
            "rmse_val": rmse(y_val, pred_val),
            "mae_val": mean_absolute_error(y_val, pred_val),
            "r2_val": r2_score(y_val, pred_val),
            "bias_val": bias(y_val, pred_val)
        })

    df = pd.DataFrame(rows)

    # Aggregate cross-validation results
    return {
        "config": name,
        "val_rmse": df.rmse_val.mean(),
        "val_mae": df.mae_val.mean(),
        "val_r2": df.r2_val.mean(),
        "val_bias": df.bias_val.mean(),
        "avg_selected_features": df.n_selected.mean(),
        "avg_total_features": df.n_total.mean()
    }


# RUN

result = eval_single_config(CONFIG_NAME, PARAMS)
display(pd.DataFrame([result]))

print("\nFINAL RESULTS")
for k, v in result.items():
    if k != "config":
        print(f"{k}: {v:.4f}")

print(f"\nLog saved to: {LOG_FILE}")


[FE_FS65_Stacking] Fold 1/8
[FE_FS65_Stacking] Fold 2/8
[FE_FS65_Stacking] Fold 3/8
[FE_FS65_Stacking] Fold 4/8
[FE_FS65_Stacking] Fold 5/8
[FE_FS65_Stacking] Fold 6/8
[FE_FS65_Stacking] Fold 7/8
[FE_FS65_Stacking] Fold 8/8


,config,val_rmse,val_mae,val_r2,val_bias,avg_selected_features,avg_total_features
0,FE_FS65_Stacking,2117.180289,1241.484492,0.952542,94.208941,17.0,25.0



FINAL RESULTS
val_rmse: 2117.1803
val_mae: 1241.4845
val_r2: 0.9525
val_bias: 94.2089
avg_selected_features: 17.0000
avg_total_features: 25.0000

Log saved to: stacking_fe_fs_65.log


sem previous owners, log do price,age - sem feature selection

In [ ]:

# SINGLE CONFIG PIPELINE (LOG TARGET + AGE)

# Numeric variables kept as continuous inputs after preprocessing
numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]

# Features explicitly removed before modeling
DROP_FROM_MODEL = ["previousOwners"]

# Cross-Validation 
# Standard K-Fold with shuffling for robust validation
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Fixed Model Hyperparameters
params = {
    "hgb": {
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 1000,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 0.33,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 800,
        "max_depth": 20,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.7,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}
# Logging Setup 
search_results = []
best_rmse = np.inf
best_config = None

log_path = "no_fs_age_log.txt"

with open(log_path, "w", encoding="utf-8") as log_file:
    
    def log(msg: str):
        """Log to console and file (synchronous)."""
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()
    
    log("# =============================")
    log("# START OF SINGLE CONFIG PIPELINE (LOG TARGET + AGE FEATURE)")
    log("# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")
    
    # Containers for fold-level metrics
    fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
    fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
    fold_med_ae = []

    # K-FOLD LOOP
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

        # Split data
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

        # Log-transform target for training stability
        y_train_log = np.log1p(y_train)
        log(f"\n[FOLD {fold}] Processing fold...")

        # Preprocessing / Feature Engineering
        
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, state=mileage_state)
        X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        # Resolve noisy categorical values using training-only state
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val, _, _   = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val, _, _   = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val, _, _   = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val, _, _   = transform_fueltype_resolver(X_val,   fuel_state)

        # Feature engineering: age (then drop year)
        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        # Drop explicitly excluded features
        X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
        X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

        # Encoding 
        # High-cardinality categoricals: target encoding (log target)
        high_card_features = ["Brand", "model"]
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        # Low-cardinality categoricals: one-hot encoding
        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])
        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        # Concatenate categorical blocks
        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high, X_val_low], axis=1)

        # Final design matrix: numeric + categorical
        X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

        # Standardize numeric features only
        if len(numeric_features) > 0:
            scaler = StandardScaler()
            X_train_final[numeric_features] = scaler.fit_transform(X_train_final[numeric_features])
            X_val_final[numeric_features]   = scaler.transform(X_val_final[numeric_features])

        # Align columns between train/validation
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        if fold == 1:
            feature_names = list(X_train_final.columns)
            log(f"  > Features Used ({len(feature_names)}): {feature_names}")

        # Modeling

        # Base learners
        estimators = [
                ("hgb", HistGradientBoostingRegressor(**params["hgb"])),
                ("rf",  RandomForestRegressor(**params["rf"])),
                ("et",  ExtraTreesRegressor(**params["et"]))
            ]

        # Linear meta-learner for stacking
        meta_model = Ridge(alpha=3.0)

        stack = StackingRegressor(
            estimators=estimators,
            final_estimator=meta_model,
            cv=5,
            n_jobs=1
        )

        # Train on log-target
        stack.fit(X_train_final, y_train_log)

        # Inverse transform predictions to original scale
        pred_tr  = np.expm1(stack.predict(X_train_final))
        pred_val = np.expm1(stack.predict(X_val_final))

        # Metrics (Original Scale)
        mae_tr  = mean_absolute_error(y_train, pred_tr)
        rmse_tr = np.sqrt(mean_squared_error(y_train, pred_tr))
        r2_tr   = r2_score(y_train, pred_tr)
        bias_tr = np.mean(y_train - pred_tr)

        mae_val  = mean_absolute_error(y_val, pred_val)
        rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))
        r2_val   = r2_score(y_val, pred_val)
        bias_val = np.mean(y_val - pred_val)

        # Store fold metrics
        fold_maes_tr.append(mae_tr); fold_rmses_tr.append(rmse_tr); fold_r2s_tr.append(r2_tr); fold_bias_tr.append(bias_tr)
        fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
        fold_med_ae.append(median_absolute_error(y_val, pred_val))

        log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
        log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")

    # CV SUMMARY
    mean_rmse_val = np.mean(fold_rmses_val)
    mean_mae_val  = np.mean(fold_maes_val)
    mean_r2_val   = np.mean(fold_r2s_val)
    mean_bias_val = np.mean(fold_bias_val)

    mean_mae_tr   = np.mean(fold_maes_tr)
    mean_r2_tr    = np.mean(fold_r2s_tr)
    mean_bias_tr  = np.mean(fold_bias_tr)

    log("\nSUMMARY ACROSS FOLDS:")
    log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
    log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

    # Store results for comparison/export
    search_results.append({
        "config": "single_config_age",
        "val_rmse": mean_rmse_val,
        "val_mae": mean_mae_val,
        "val_r2": mean_r2_val,
        "val_bias": mean_bias_val,
        "train_mae": mean_mae_tr,
        "train_r2": mean_r2_tr,
        "train_bias": mean_bias_tr,
        "val_med_ae": np.mean(fold_med_ae)
    })

    log("# END OF PIPELINE")
    
# Final results table
results_df = pd.DataFrame(search_results)
display(results_df)


# =============================
# START OF SINGLE CONFIG PIPELINE (LOG TARGET + AGE FEATURE)
# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS
# LOG FILE SAVED TO: no_fs_age_log.txt
# =============================

[FOLD 1] Processing fold...
  > Features Used (15): ['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
  > [TRAIN] R2: 0.9750 | RMSE: 1543 | MAE: 892 | Bias: 56.3
  > [VAL]   R2: 0.9601 | RMSE: 1901 | MAE: 1212 | Bias: 44.3

[FOLD 2] Processing fold...
  > [TRAIN] R2: 0.9763 | RMSE: 1500 | MAE: 923 | Bias: 55.3
  > [VAL]   R2: 0.9533 | RMSE: 2094 | MAE: 1246 | Bias: 82.0

[FOLD 3] Processing fold...
  > [TRAIN] R2: 0.9769 | RMSE: 1481 | MAE: 887 | Bias: 53.7
  > [VAL]   R2: 0.9502 | RMSE: 2163 | MAE: 1242 | Bias: 128.7

[FOLD 4] Processing fold...
  > [TRAIN] R2: 0.9788 | RMSE: 1410 | MAE: 903 |

,config,val_rmse,val_mae,val_r2,val_bias,train_mae,train_r2,train_bias,val_med_ae
0,single_config_age,2098.159082,1225.830306,0.953369,92.070382,901.421753,0.976953,54.369274,774.888285


sem previous owners, log do price,age - com fs -80 %

In [ ]:

# SINGLE CONFIG PIPELINE (LOG TARGET + AGE + FS 80%)

# Numeric variables kept as continuous inputs after preprocessing
numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]

# Explicitly removed feature (low signal / high noise)
DROP_FROM_MODEL = ["previousOwners"]

# Ratio of features kept after feature selection
FS_KEEP_RATIO = 0.80

# CROSS VALIDATION SETUP
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# FEATURE SELECTION MODEL
# RandomForest is used only to rank feature importances
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# FIXED MODEL PARAMETERS
# No hyperparameter search in this experiment
params = {
    "hgb": {
        "max_iter": 600,
        "learning_rate": 0.1,
        "max_depth": 15,
        "max_leaf_nodes": 127,
        "min_samples_leaf": 20,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 1000,
        "min_samples_split":2,
        "min_samples_leaf": 2,
        "max_features": 0.33,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 800,
        "max_depth": 20,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.7,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}


search_results = []
best_rmse = np.inf
best_config = None

# LOGGING SETUP
log_path ="mean_age_fs80_log.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# START OF SINGLE CONFIG PIPELINE (LOG TARGET + AGE + FS 80%)")
    log("# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")

    # METRIC CONTAINERS
    fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
    fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
    fold_med_ae = []
    fold_nsel = []

    # K FOLD LOOP
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        
        # DATA SPLIT
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

        # LOG TARGET TRANSFORMATION
        y_train_log = np.log1p(y_train)
        log(f"\n[FOLD {fold}] Processing fold...")

        # PREPROCESSING AND FEATURE ENGINEERING

        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, state=mileage_state)
        X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val, _, _   = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val, _, _   = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val, _, _   = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val, _, _   = transform_fueltype_resolver(X_val,   fuel_state)

        # AGE FEATURE ENGINEERING
        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        # DROP EXCLUDED FEATURES
        X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
        X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

        # ENCODING
        high_card_features = ["Brand", "model"]
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])
        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high, X_val_low], axis=1)

        X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

        # SCALE NUMERIC FEATURES ONLY
        if len(numeric_features) > 0:
            scaler = StandardScaler()
            X_train_final[numeric_features] = scaler.fit_transform(X_train_final[numeric_features])
            X_val_final[numeric_features]   = scaler.transform(X_val_final[numeric_features])

        # ALIGN TRAIN AND VALIDATION MATRICES
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        if fold == 1:
            feature_names = list(X_train_final.columns)
            log(f"  > Features before FS ({len(feature_names)}): {feature_names}")

        # FEATURE SELECTION
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        selected_cols = X_train_final.columns[selector.get_support()]
        fold_nsel.append(len(selected_cols))

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        if fold == 1:
            log(f"  > Selected features ({len(selected_cols)}/{n_feats}) with FS_KEEP_RATIO={FS_KEEP_RATIO:.2f}")

        # STACKING MODEL
        estimators = [
                ("hgb", HistGradientBoostingRegressor(**params["hgb"])),
                ("rf",  RandomForestRegressor(**params["rf"])),
                ("et",  ExtraTreesRegressor(**params["et"]))
            ]

        meta_model = Ridge(alpha=3.0)

        stack = StackingRegressor(
            estimators=estimators,
            final_estimator=meta_model,
            cv=5,
            n_jobs=1
        )

        # TRAIN ON LOG TARGET
        stack.fit(X_train_final, y_train_log)

        # PREDICTIONS ON ORIGINAL SCALE
        pred_tr  = np.expm1(stack.predict(X_train_final))
        pred_val = np.expm1(stack.predict(X_val_final))

        # METRICS
        mae_tr  = mean_absolute_error(y_train, pred_tr)
        rmse_tr = np.sqrt(mean_squared_error(y_train, pred_tr))
        r2_tr   = r2_score(y_train, pred_tr)
        bias_tr = np.mean(y_train - pred_tr)

        mae_val  = mean_absolute_error(y_val, pred_val)
        rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))
        r2_val   = r2_score(y_val, pred_val)
        bias_val = np.mean(y_val - pred_val)

        fold_maes_tr.append(mae_tr); fold_rmses_tr.append(rmse_tr); fold_r2s_tr.append(r2_tr); fold_bias_tr.append(bias_tr)
        fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
        fold_med_ae.append(median_absolute_error(y_val, pred_val))

        log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
        log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")

    # CROSS VALIDATION SUMMARY
    mean_rmse_val = np.mean(fold_rmses_val)
    mean_mae_val  = np.mean(fold_maes_val)
    mean_r2_val   = np.mean(fold_r2s_val)
    mean_bias_val = np.mean(fold_bias_val)

    mean_mae_tr   = np.mean(fold_maes_tr)
    mean_r2_tr    = np.mean(fold_r2s_tr)
    mean_bias_tr  = np.mean(fold_bias_tr)
    mean_nsel     = float(np.mean(fold_nsel))

    log("\nSUMMARY ACROSS FOLDS:")
    log(f"  Selected features (avg): {mean_nsel:.0f} / {X_train_final.shape[1]}")
    log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
    log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

    search_results.append({
        "config": "single_config_age_fs90",
        "fs_keep_ratio": FS_KEEP_RATIO,
        "avg_selected_features": mean_nsel,
        "val_rmse": mean_rmse_val,
        "val_mae": mean_mae_val,
        "val_r2": mean_r2_val,
        "val_bias": mean_bias_val,
        "train_mae": mean_mae_tr,
        "train_r2": mean_r2_tr,
        "train_bias": mean_bias_tr,
        "val_med_ae": np.mean(fold_med_ae)
    })

    log("# END OF PIPELINE")

# FINAL RESULTS TABLE
results_df = pd.DataFrame(search_results)
display(results_df)


# =============================
# START OF SINGLE CONFIG PIPELINE (LOG TARGET + AGE + FS 80%)
# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS
# LOG FILE SAVED TO: mean_age_fs80_log.txt
# =============================

[FOLD 1] Processing fold...
  > Features before FS (15): ['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
  > Selected features (12/15) with FS_KEEP_RATIO=0.80
  > [TRAIN] R2: 0.9721 | RMSE: 1631 | MAE: 954 | Bias: 62.3
  > [VAL]   R2: 0.9590 | RMSE: 1926 | MAE: 1226 | Bias: 49.4

[FOLD 2] Processing fold...
  > [TRAIN] R2: 0.9765 | RMSE: 1493 | MAE: 918 | Bias: 54.3
  > [VAL]   R2: 0.9538 | RMSE: 2083 | MAE: 1242 | Bias: 74.0

[FOLD 3] Processing fold...
  > [TRAIN] R2: 0.9753 | RMSE: 1532 | MAE: 920 | Bias: 57.9
  > [VAL]   R2: 0.9501 | RMSE: 2166 | MAE: 1247 | Bias: 128.6

[FOLD 4] Pro

,config,fs_keep_ratio,avg_selected_features,val_rmse,val_mae,val_r2,val_bias,train_mae,train_r2,train_bias,val_med_ae
0,single_config_age_fs90,0.8,12.0,2110.265393,1236.315902,0.952833,92.182984,946.41033,0.974735,58.512953,781.654205


submissao final do melhor modelo e kaggle submission 

In [10]:

# STACKING SUBMISSION — LOG PRICE / NO previousOwners

print("- STACKING submission (log-price, no previousOwners)")


# CONFIG

DROP_FROM_MODEL = ["previousOwners"]

# Numeric features exactly as used in CV
numeric_features = ["mileage", "engineSize", "tax", "mpg", "year"]

# High cardinality categorical features
high_card_features = ["Brand", "model"]

# STACKING PARAMETERS

# Same hyperparameters validated during CV
STACK_PARAMS = {
    "hgb": {
        "max_iter": 1000,
        "learning_rate": 0.1,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 20,
        "l2_regularization": 3.0,
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 1000,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 0.33,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 800,
        "max_depth": 20,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.7,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}


# LOAD TEST DATA

try:
    test_df_raw = pd.read_csv("test.csv")
except:
    test_df_raw = pd.read_csv("../../project_data/test.csv")

# Identify ID column dynamically
ID_COL = next((c for c in ["id", "carID", "carId", "car_id"] if c in test_df_raw.columns), None)

if ID_COL:
    test_ids = test_df_raw[ID_COL]
    X_test = test_df_raw.drop(columns=[ID_COL])
else:
    test_ids = test_df_raw.index
    X_test = test_df_raw.copy()

# FULL TRAIN DATA
X_full = X.copy()
y_full = y.copy()

# Log transform target for training
y_full_log = np.log1p(y_full)

# Low cardinality categorical features
low_card_features = [c for c in categorical_features if c not in high_card_features]

# STRING NORMALIZATION
# Ensure consistent categorical representation between train and test
for col in ["Brand", "model", "transmission", "fuelType"]:
    if col in X_full:
        X_full[col] = fill_unknown(X_full[col])
        X_full = column_string_transformer(X_full, col)
    if col in X_test:
        X_test[col] = fill_unknown(X_test[col])
        X_test = column_string_transformer(X_test, col)


# FIT ALL TRANSFORMS ON FULL TRAIN
# States are reused for test to avoid leakage

year_state = fit_year_median(X_full, "year", "model")
X_full = transform_year_with_model_median(X_full, year_state)

mileage_state = fit_mileage_imputer(X_full, "mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, mileage_state)

engine_state = fit_engine_size_imputer(X_full, "engineSize")
X_full = transform_engine_size_imputer(X_full, engine_state)

X_full = transform_tax_custom_rules(X_full, "tax", "year", "fuelType", "engineSize")

mpg_state = fit_mpg_imputer(X_full, "mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, mpg_state)

owners_state = fit_previous_owners_imputer(X_full, "previousOwners", "year", "mileage")
X_full = transform_previous_owners_imputer(X_full, owners_state)

brand_state = fit_ambiguous_brand_resolver(X_full, valid_brands)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(X_full, valid_models_by_brand)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(X_full, valid_transmissions)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(X_full, valid_fueltypes)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# DROP PREVIOUSOWNERS 
X_full = X_full.drop(columns=DROP_FROM_MODEL, errors="ignore")

# ENCODING
# Target encoding for high cardinality features
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)
X_high = te.transform(X_full[high_card_features])

# One hot encoding for low cardinality features
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])
X_low = ohe.transform(X_full[low_card_features])

# FINAL TRAIN MATRIX
X_train_final = pd.concat(
    [X_full[numeric_features], X_high, X_low],
    axis=1
)

# SCALE NUMERIC FEATURES
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)


# STACKING MODEL 
estimators = [
    ("hgb", HistGradientBoostingRegressor(**STACK_PARAMS["hgb"])),
    ("rf",  RandomForestRegressor(**STACK_PARAMS["rf"])),
    ("et",  ExtraTreesRegressor(**STACK_PARAMS["et"]))
]

# Ridge meta learner
stack = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=4.0),
    cv=5,
    passthrough=False,
    n_jobs=1
)

print("- training stacking (log-price)")
stack.fit(X_train_scaled, y_full_log)

# TRANSFORM TEST DATA

X_test = transform_year_with_model_median(X_test, year_state)
X_test = transform_mileage_imputer(X_test, mileage_state)
X_test = transform_engine_size_imputer(X_test, engine_state)
X_test = transform_tax_custom_rules(X_test, "tax", "year", "fuelType", "engineSize")
X_test = transform_mpg_imputer(X_test, mpg_state)
X_test = transform_previous_owners_imputer(X_test, owners_state)

X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

X_test = X_test.drop(columns=DROP_FROM_MODEL, errors="ignore")

X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])

X_test_final = pd.concat(
    [X_test[numeric_features], X_test_high, X_test_low],
    axis=1
)

# ALIGN WITH TRAIN FEATURES
X_test_final = X_test_final.reindex(columns=X_train_final.columns, fill_value=0)
X_test_scaled = scaler.transform(X_test_final)

# PREDICTION AND SUBMISSION

pred_log = stack.predict(X_test_scaled)
pred = np.expm1(pred_log)
pred = np.maximum(pred, 0)

submission = pd.DataFrame({
    ID_COL if ID_COL else "id": test_ids,
    "price": pred
})

submission.to_csv("submission_stacking_final.csv", index=False)
print("- submission saved")
print(submission.head())


- STACKING submission (log-price, no previousOwners)
- training stacking (log-price)
- submission saved
    carID         price
0   89856   9857.837383
1  106581  22414.667064
2   80886  13810.573634
3  100174  16969.665067
4   81376  22987.838439


graficos

In [ ]:
# -----------------------------
# 1) Best config (your chosen one)
# -----------------------------
best_params = {
    "hgb": {
        "max_iter": 1000,
        "learning_rate": 0.1,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 20,
        "l2_regularization": 3.0,
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 1000,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 0.33,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 800,
        "max_depth": 20,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.7,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

MODEL_NAME = "STACKING (no previousOwners, log(price), no FE, no FS)"

# -----------------------------
# 2) Build FULL training matrix for this setting
#    (same logic as the CV loop, but fit on ALL data)
# -----------------------------
numeric_features = ["year", "mileage", "engineSize", "tax", "mpg"]  # previousOwners excluded
categorical_features = ["Brand", "model", "transmission", "fuelType"]

X_full = X.copy()
y_full = y.copy()

# ensure previousOwners never enters the pipeline
X_full = X_full.drop(columns=["previousOwners"], errors="ignore")

# log target (train in log-space)
y_full_log = np.log1p(y_full)

# ---- Preprocessing (fit on full data, transform on full data) ----
year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

mileage_state = fit_mileage_imputer(X_full, mileage_col="mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full,
    valid_brands=valid_brands,
    brand_col="Brand",
    model_col="model",
    year_col="year",
)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(
    train_df=X_full,
    valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand",
    model_col="model",
    year_col="year",
    fuel_col="fuelType",
    mpg_col="mpg",
)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(
    train_df=X_full,
    valid_transmissions=valid_transmissions,
    transm_col="transmission",
    brand_col="Brand",
    model_col="model",
    fuel_col="fuelType",
)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(
    train_df=X_full,
    valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType",
    brand_col="Brand",
    model_col="model",
    transm_col="transmission",
)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

X_full = transform_tax_custom_rules(X_full, "tax", "year", "fuelType", "engineSize")

# ---- Encoding (TE for Brand+model, OHE for the rest) ----
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full_log)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])
X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

# ---- Numeric scaling (keep it for consistency with your pipeline) ----
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{c}" for c in numeric_features],
)

# final matrix used for training and for your plots
X_full_sel = pd.concat([X_full_num_df, X_full_cat], axis=1)

# -----------------------------
# 3) Fit FINAL model on FULL data (log-space)
# -----------------------------
estimators = [
    ("hgb", HistGradientBoostingRegressor(**best_params["hgb"])),
    ("rf",  RandomForestRegressor(**best_params["rf"])),
    ("et",  ExtraTreesRegressor(**best_params["et"]))
]

meta_model = Ridge(alpha=3.0)

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=meta_model,
    cv=5,
    n_jobs=1
)

print("- training stacking (log-price)")
stack.fit(X_full_sel, y_full_log)

# -----------------------------
# 4) Run ALL visualizations for this best model
# -----------------------------
plot_pred_vs_true_fs(stack, MODEL_NAME)
plot_residuals_fs(stack, MODEL_NAME)
plot_residual_distribution_fs(stack, MODEL_NAME)

KeyboardInterrupt: 

In [ ]:
plot_permutation_importance_fs(stack, MODEL_NAME, n_repeats=5, top=20)

In [ ]:
apply_shap_fs(stack, X_full_sel, model_name=MODEL_NAME, sample_size=2000)

In [ ]:
plot_feature_importance_fs(stack, MODEL_NAME, top=20)

In [ ]:
split_freq_df = compute_split_frequency_fs(stack)
display(split_freq_df.head(30))